# 02 Driver Analysis

This notebook analyzes driver-level competitive patterns in the Silver layer: win concentration, lap-time consistency, overtaking balance, grid-to-finish dynamics, teammate gaps, and year-over-year performance trends.

The analysis uses only operational Silver tables and avoids telemetry-scale data.

In [1]:
from pathlib import Path
import sys
from datetime import datetime
import json

import pandas as pd
import numpy as np
import plotly.express as px
import pyarrow.parquet as pq

ROOT = Path.cwd()
while not (ROOT / "configs" / "pipeline_config.yaml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

SHARED = ROOT / "eda" / "shared" / "scripts"
if str(SHARED) not in sys.path:
    sys.path.insert(0, str(SHARED))

from config import CLEANED_DATA_PATH

NOTEBOOK_NAME = "02_driver_analysis"
OUTPUT_TABLES = ROOT / "eda" / "silver" / "outputs" / "tables" / NOTEBOOK_NAME
OUTPUT_CHARTS = ROOT / "eda" / "silver" / "outputs" / "charts" / NOTEBOOK_NAME
OUTPUT_REPORTS = ROOT / "eda" / "silver" / "outputs" / "reports" / NOTEBOOK_NAME
INSIGHTS = ROOT / "eda" / "silver" / "insights"
CHECKPOINTS = ROOT / "eda" / "silver" / "checkpoints"
for path in [OUTPUT_TABLES, OUTPUT_CHARTS, OUTPUT_REPORTS, INSIGHTS, CHECKPOINTS]:
    path.mkdir(parents=True, exist_ok=True)

def write_report(name: str, payload: dict) -> None:
    (OUTPUT_REPORTS / f"{name}.json").write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")

def write_insight(title: str, observations: list[str], issues: list[str], recommendations: list[str]) -> None:
    content = f"# {title}\n\n"
    content += f"**Generated at:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n"
    content += "## Key Observations\n\n" + "\n".join(f"- {item}" for item in observations) + "\n\n"
    content += "## Issues\n\n" + ("\n".join(f"- {item}" for item in issues) if issues else "- None") + "\n\n"
    content += "## Recommendations\n\n" + "\n".join(f"- {item}" for item in recommendations) + "\n"
    (INSIGHTS / f"{NOTEBOOK_NAME}.md").write_text(content, encoding="utf-8")

print("=" * 72)
print(f"SILVER EDA - {NOTEBOOK_NAME}")
print(f"Start time: {datetime.now()}")
print(f"Cleaned data path: {CLEANED_DATA_PATH}")
print("=" * 72)


SILVER EDA - 02_driver_analysis
Start time: 2026-06-02 00:45:58.166941
Cleaned data path: D:\F1_WinRate_Predictor\data\cleaned


In [2]:
drivers = pd.read_parquet(CLEANED_DATA_PATH / "drivers.parquet")
session_result = pd.read_parquet(CLEANED_DATA_PATH / "session_result.parquet")
laps = pd.read_parquet(CLEANED_DATA_PATH / "laps.parquet")
overtakes = pd.read_parquet(CLEANED_DATA_PATH / "overtakes.parquet")
starting_grid = pd.read_parquet(CLEANED_DATA_PATH / "starting_grid.parquet")
sessions = pd.read_parquet(CLEANED_DATA_PATH / "sessions.parquet")

sessions["event_type"] = np.where(
    sessions["session_name"].astype(str).str.lower().eq("sprint"),
    "SPRINT_RACE",
    "GRAND_PRIX_RACE",
)
driver_dim = (
    drivers[["session_key", "driver_number", "full_name", "team_name"]]
    .drop_duplicates(["session_key", "driver_number"])
)
results = (
    session_result
    .merge(sessions[["session_key", "year", "event_type"]], on="session_key", how="left")
    .merge(driver_dim, on=["session_key", "driver_number"], how="left")
)
results["driver_id"] = results["full_name"].fillna("Driver " + results["driver_number"].astype(str))
results["finish_pos"] = pd.to_numeric(results["position"], errors="coerce")
results["is_classified"] = results["finish_pos"].notna()
print(f"Drivers: {results['driver_id'].nunique()}")
print(f"Sessions: {sessions['session_key'].nunique()} ({sessions['event_type'].value_counts().to_dict()})")
print(f"Result rows: {len(results):,}")
print(f"Laps: {len(laps):,}")
print(f"Overtakes: {len(overtakes):,}")

Drivers: 28
Sessions: 70 ({'GRAND_PRIX_RACE': 55, 'SPRINT_RACE': 15})
Result rows: 1,374
Laps: 63,676
Overtakes: 13,985


## 1. Win Rate and Competitive Concentration

Formula 1 wins are usually concentrated among a small set of drivers. This section quantifies how concentrated wins are in the cleaned race and sprint sample.

In [3]:
driver_summary = (
    results.groupby("driver_id", as_index=False)
    .agg(
        starts=("session_key", "nunique"),
        classified_finishes=("is_classified", "sum"),
        wins=("finish_pos", lambda s: int((s == 1).sum())),
        podiums=("finish_pos", lambda s: int((s <= 3).sum())),
        top10s=("finish_pos", lambda s: int((s <= 10).sum())),
        avg_finish=("finish_pos", "mean"),
        points=("points", "sum"),
        primary_driver_number=("driver_number", lambda s: int(s.mode().iloc[0]) if not s.mode().empty else int(s.iloc[0])),
        primary_team=("team_name", lambda s: s.dropna().mode().iloc[0] if not s.dropna().mode().empty else "Unknown"),
    )
)
driver_summary["win_rate_pct"] = np.where(driver_summary["starts"] > 0, driver_summary["wins"] / driver_summary["starts"] * 100, 0)
driver_summary["podium_rate_pct"] = np.where(driver_summary["starts"] > 0, driver_summary["podiums"] / driver_summary["starts"] * 100, 0)
driver_summary["top10_rate_pct"] = np.where(driver_summary["starts"] > 0, driver_summary["top10s"] / driver_summary["starts"] * 100, 0)
driver_summary = driver_summary.sort_values(["wins", "points", "avg_finish"], ascending=[False, False, True])
driver_summary.to_csv(OUTPUT_TABLES / "driver_result_summary.csv", index=False)
display(driver_summary.head(20))

,driver_id,starts,classified_finishes,wins,podiums,top10s,avg_finish,points,primary_driver_number,primary_team,win_rate_pct,podium_rate_pct,top10_rate_pct
19,Max VERSTAPPEN,68,65,23,37,64,3.492308,901.0,1,Red Bull Racing,33.823529,54.411765,94.117647
15,Lando NORRIS,68,63,15,42,60,3.746032,855.0,4,McLaren,22.058824,61.764706,88.235294
22,Oscar PIASTRI,68,62,11,34,60,3.983871,750.0,81,McLaren,16.176471,50.000000,88.235294
9,George RUSSELL,68,65,7,21,61,4.830769,652.0,63,Mercedes,10.294118,30.882353,89.705882
13,Kimi ANTONELLI,38,34,4,10,27,6.794118,281.0,12,Mercedes,10.526316,26.315789,71.052632
3,Charles LECLERC,68,63,3,26,59,4.523810,673.0,16,Ferrari,4.411765,38.235294,86.764706
16,Lewis HAMILTON,68,63,3,11,56,6.460317,451.0,44,Ferrari,4.411765,16.176471,82.352941
2,Carlos SAINZ,67,60,2,13,43,8.016667,360.0,55,Williams,2.985075,19.402985,64.179104
24,Sergio PEREZ,38,33,0,6,21,9.757576,152.0,11,Red Bull Racing,0.000000,15.789474,55.263158
6,Fernando ALONSO,68,56,0,0,27,11.107143,126.0,14,Aston Martin,0.000000,0.000000,39.705882


In [4]:
qualified = driver_summary[driver_summary["starts"] >= 5].copy()
win_rates = qualified["win_rate_pct"]
total_wins = float(driver_summary["wins"].sum())
top1_pct = float(driver_summary.head(1)["wins"].sum() / total_wins * 100) if total_wins else 0.0
top3_pct = float(driver_summary.head(3)["wins"].sum() / total_wins * 100) if total_wins else 0.0
top5_pct = float(driver_summary.head(5)["wins"].sum() / total_wins * 100) if total_wins else 0.0

fig = px.bar(
    driver_summary.head(20).sort_values("wins"),
    x="wins",
    y="driver_id",
    orientation="h",
    color="win_rate_pct",
    title="Top Drivers by Wins and Win Rate",
    labels={"driver_id": "Driver", "wins": "Wins", "win_rate_pct": "Win rate %"},
)
fig.write_html(OUTPUT_CHARTS / "driver_wins.html", include_plotlyjs="cdn")
fig.show()

In [5]:
lorenz = driver_summary.sort_values("wins", ascending=True).copy()
lorenz["cumulative_wins_pct"] = lorenz["wins"].cumsum() / total_wins * 100 if total_wins else 0
lorenz["drivers_pct"] = (np.arange(len(lorenz)) + 1) / len(lorenz) * 100 if len(lorenz) else []
lorenz.to_csv(OUTPUT_TABLES / "win_concentration_lorenz.csv", index=False)
fig = px.line(lorenz, x="drivers_pct", y="cumulative_wins_pct", title="Win Concentration Lorenz Curve")
fig.add_scatter(x=[0, 100], y=[0, 100], mode="lines", name="Equal distribution")
fig.write_html(OUTPUT_CHARTS / "win_concentration.html", include_plotlyjs="cdn")
fig.show()

## 2. Lap-Time Consistency vs Finishing Outcome

A stable driver profile should show lower lap-time variance and stronger average finishing outcomes. This check aggregates completed laps by driver and compares consistency with average finish position.

In [6]:
lap_base = laps[laps["lap_duration"].notna()].copy()
lap_base["lap_duration"] = pd.to_numeric(lap_base["lap_duration"], errors="coerce")
lap_base = lap_base[lap_base["lap_duration"].between(50, 900)]
lap_base = lap_base.merge(driver_dim, on=["session_key", "driver_number"], how="left")
lap_base["driver_id"] = lap_base["full_name"].fillna("Driver " + lap_base["driver_number"].astype(str))
consistency = (
    lap_base.groupby("driver_id", as_index=False)
    .agg(
        avg_lap_time=("lap_duration", "mean"),
        std_lap_time=("lap_duration", "std"),
        median_lap_time=("lap_duration", "median"),
        lap_count=("lap_duration", "count"),
        outlier_laps=("is_outlier_lap", "sum"),
        primary_driver_number=("driver_number", lambda s: int(s.mode().iloc[0]) if not s.mode().empty else int(s.iloc[0])),
        primary_team=("team_name", lambda s: s.dropna().mode().iloc[0] if not s.dropna().mode().empty else "Unknown"),
    )
)
avg_finish = results.groupby("driver_id", as_index=False).agg(avg_finish=("finish_pos", "mean"), starts=("session_key", "nunique"))
consistency = consistency.merge(avg_finish, on="driver_id", how="left")
consistency = consistency[(consistency["lap_count"] >= 100) & consistency["avg_finish"].notna()]
consistency_corr = float(consistency["std_lap_time"].corr(consistency["avg_finish"])) if len(consistency) > 1 else float("nan")
consistency.to_csv(OUTPUT_TABLES / "driver_lap_consistency.csv", index=False)
display(consistency.sort_values("std_lap_time").head(20))

,driver_id,avg_lap_time,std_lap_time,median_lap_time,lap_count,outlier_laps,primary_driver_number,primary_team,avg_finish,starts
11,Jack DOOHAN,98.851359,11.693441,97.1920,309,12,7,Alpine,15.714286,9
1,Arvid LINDBLAD,96.123739,13.825236,95.8170,253,12,41,Racing Bulls,11.200000,8
12,Kevin MAGNUSSEN,92.041581,13.891228,90.8110,1330,39,20,Haas F1 Team,12.615385,27
25,Valtteri BOTTAS,92.505751,13.984892,91.6880,1769,61,77,Kick Sauber,15.764706,38
4,Daniel RICCIARDO,90.647461,14.157826,86.1165,1056,32,3,RB,12.263158,21
24,Sergio PEREZ,92.471376,14.242219,92.5575,1698,56,11,Red Bull Racing,9.757576,38
27,ZHOU Guanyu,91.860832,14.519885,88.1720,1461,45,24,Kick Sauber,15.357143,30
18,Logan SARGEANT,90.606374,14.632519,87.4200,827,22,2,Williams,16.266667,17
7,Franco COLAPINTO,91.550716,14.809146,87.6290,1910,69,43,Alpine,13.972973,42
2,Carlos SAINZ,91.553383,15.071699,89.4125,3134,102,55,Williams,8.016667,67


In [7]:
fig = px.scatter(
    consistency,
    x="std_lap_time",
    y="avg_finish",
    size="lap_count",
    hover_name="driver_id",
    title=f"Lap-Time Consistency vs Average Finish (corr={consistency_corr:.3f})",
    labels={"std_lap_time": "Lap duration standard deviation", "avg_finish": "Average finish position"},
)
if len(consistency) > 2:
    fit = np.polyfit(consistency["std_lap_time"], consistency["avg_finish"], deg=1)
    x_line = np.array([consistency["std_lap_time"].min(), consistency["std_lap_time"].max()])
    fig.add_scatter(x=x_line, y=fit[0] * x_line + fit[1], mode="lines", name="Linear fit")
fig.update_yaxes(autorange="reversed")
fig.write_html(OUTPUT_CHARTS / "consistency_vs_finish.html", include_plotlyjs="cdn")
fig.show()

## 3. Overtaking Balance

Overtake balance measures racecraft and traffic exposure. A positive net value means the driver made more overtakes than they received.

In [8]:
overtaker_dim = driver_dim.rename(columns={"driver_number": "overtaking_driver_number", "full_name": "overtaking_driver_id"})
overtaken_dim = driver_dim.rename(columns={"driver_number": "overtaken_driver_number", "full_name": "overtaken_driver_id"})
overtakes_named = (
    overtakes
    .merge(overtaker_dim[["session_key", "overtaking_driver_number", "overtaking_driver_id"]], on=["session_key", "overtaking_driver_number"], how="left")
    .merge(overtaken_dim[["session_key", "overtaken_driver_number", "overtaken_driver_id"]], on=["session_key", "overtaken_driver_number"], how="left")
)
overtakes_named["overtaking_driver_id"] = overtakes_named["overtaking_driver_id"].fillna("Driver " + overtakes_named["overtaking_driver_number"].astype(str))
overtakes_named["overtaken_driver_id"] = overtakes_named["overtaken_driver_id"].fillna("Driver " + overtakes_named["overtaken_driver_number"].astype(str))
made = overtakes_named.groupby("overtaking_driver_id").size().reset_index(name="overtakes_made").rename(columns={"overtaking_driver_id": "driver_id"})
received = overtakes_named.groupby("overtaken_driver_id").size().reset_index(name="overtakes_received").rename(columns={"overtaken_driver_id": "driver_id"})
overtake_stats = made.merge(received, on="driver_id", how="outer").fillna(0)
overtake_stats["net_overtakes"] = overtake_stats["overtakes_made"] - overtake_stats["overtakes_received"]
overtake_stats["overtake_ratio"] = overtake_stats["overtakes_made"] / (overtake_stats["overtakes_received"] + 1)
overtake_stats = overtake_stats.sort_values("net_overtakes", ascending=False)
overtake_stats.to_csv(OUTPUT_TABLES / "driver_overtake_balance.csv", index=False)
display(overtake_stats.head(20))

,driver_id,overtakes_made,overtakes_received,net_overtakes,overtake_ratio
16,Lewis HAMILTON,671,571,100,1.173077
5,Esteban OCON,848,764,84,1.108497
21,Oliver BEARMAN,512,429,83,1.190698
17,Liam LAWSON,551,479,72,1.147917
27,ZHOU Guanyu,339,285,54,1.185315
12,Kevin MAGNUSSEN,361,325,36,1.107362
7,Franco COLAPINTO,417,394,23,1.055696
3,Charles LECLERC,546,523,23,1.041985
14,Lance STROLL,772,754,18,1.022517
24,Sergio PEREZ,376,360,16,1.041551


In [9]:
plot_overtakes = overtake_stats.head(20).sort_values("net_overtakes")
fig = px.bar(
    plot_overtakes,
    x="net_overtakes",
    y="driver_id",
    orientation="h",
    color="net_overtakes",
    title="Top Net Overtake Balance by Driver",
)
fig.write_html(OUTPUT_CHARTS / "net_overtakes.html", include_plotlyjs="cdn")
fig.show()

In [10]:
fig = px.scatter(
    overtake_stats,
    x="overtakes_made",
    y="overtakes_received",
    hover_name="driver_id",
    color="net_overtakes",
    title="Overtakes Made vs Received",
)
max_axis = max(overtake_stats["overtakes_made"].max(), overtake_stats["overtakes_received"].max())
fig.add_scatter(x=[0, max_axis], y=[0, max_axis], mode="lines", name="Equal balance")
fig.write_html(OUTPUT_CHARTS / "overtakes_made_vs_received.html", include_plotlyjs="cdn")
fig.show()

## 4. Grid-to-Finish Dynamics

Starting position is one of the strongest pre-race predictors. This section quantifies position change and how often wins come from different grid slots.

In [11]:
grid_finish = starting_grid.merge(
    session_result[["session_key", "driver_number", "position"]],
    on=["session_key", "driver_number"],
    suffixes=("_grid", "_finish"),
)
grid_finish = grid_finish.merge(driver_dim, on=["session_key", "driver_number"], how="left")
grid_finish["driver_id"] = grid_finish["full_name"].fillna("Driver " + grid_finish["driver_number"].astype(str))
grid_finish["grid_pos"] = pd.to_numeric(grid_finish["position_grid"], errors="coerce")
grid_finish["finish_pos"] = pd.to_numeric(grid_finish["position_finish"], errors="coerce")
grid_finish = grid_finish.dropna(subset=["grid_pos", "finish_pos"])
grid_finish["position_change"] = grid_finish["grid_pos"] - grid_finish["finish_pos"]
grid_corr = float(grid_finish["grid_pos"].corr(grid_finish["finish_pos"])) if len(grid_finish) > 1 else float("nan")
grid_finish.to_csv(OUTPUT_TABLES / "grid_finish_dynamics.csv", index=False)
display(grid_finish.sort_values("position_change", ascending=False).head(20))

,position_grid,driver_number,lap_duration,meeting_key,session_key,session_type,position_finish,full_name,team_name,driver_id,grid_pos,finish_pos,position_change
1057,19,27,86.574,1277,9947,Race,3.0,Nico HULKENBERG,Kick Sauber,Nico HULKENBERG,19,3.0,16.0
593,18,44,95.573,1233,9672,Race,2.0,Lewis HAMILTON,Mercedes,Lewis HAMILTON,18,2.0,16.0
557,19,16,83.833,1252,9662,Race,3.0,Charles LECLERC,Ferrari,Charles LECLERC,19,3.0,16.0
117,20,3,88.617,1234,9506,Race,4.0,Daniel RICCIARDO,RB,Daniel RICCIARDO,20,4.0,16.0
472,17,1,87.771,1249,9636,Race,1.0,Max VERSTAPPEN,Red Bull Racing,Max VERSTAPPEN,17,1.0,16.0
477,20,55,89.406,1249,9635,Race,5.0,Carlos SAINZ,Ferrari,Carlos SAINZ,20,5.0,15.0
417,20,63,92.974,1247,9616,Race,5.0,George RUSSELL,Mercedes,George RUSSELL,20,5.0,15.0
418,20,63,92.974,1247,9617,Race,6.0,George RUSSELL,Mercedes,George RUSSELL,20,6.0,14.0
795,17,12,116.314,1274,9858,Race,3.0,Kimi ANTONELLI,Mercedes,Kimi ANTONELLI,17,3.0,14.0
1217,20,3,NaN,1279,11234,Race,6.0,Max VERSTAPPEN,Red Bull Racing,Max VERSTAPPEN,20,6.0,14.0


In [12]:
grid_finish["grid_bracket"] = pd.cut(
    grid_finish["grid_pos"],
    bins=[0, 3, 6, 10, 15, 25],
    labels=["P1-P3", "P4-P6", "P7-P10", "P11-P15", "P16+"],
)
bracket_stats = grid_finish.groupby("grid_bracket", observed=False).agg(
    avg_position_change=("position_change", "mean"),
    median_position_change=("position_change", "median"),
    drivers=("driver_number", "count"),
).reset_index()
bracket_stats.to_csv(OUTPUT_TABLES / "grid_bracket_position_change.csv", index=False)
display(bracket_stats)

,grid_bracket,avg_position_change,median_position_change,drivers
0,P1-P3,-1.140625,0.0,192
1,P4-P6,-0.635417,0.0,192
2,P7-P10,-0.563265,0.0,245
3,P11-P15,0.705479,1.0,292
4,P16+,4.108553,3.0,304


In [13]:
fig = px.scatter(
    grid_finish,
    x="grid_pos",
    y="finish_pos",
    color="position_change",
    hover_name="driver_id",
    title=f"Grid Position vs Finish Position (corr={grid_corr:.3f})",
)
fig.update_yaxes(autorange="reversed")
fig.write_html(OUTPUT_CHARTS / "grid_vs_finish_driver.html", include_plotlyjs="cdn")
fig.show()

In [14]:
fig = px.bar(
    bracket_stats,
    x="grid_bracket",
    y="avg_position_change",
    title="Average Position Change by Starting Bracket",
    labels={"avg_position_change": "Average positions gained", "grid_bracket": "Grid bracket"},
)
fig.write_html(OUTPUT_CHARTS / "grid_bracket_position_change.html", include_plotlyjs="cdn")
fig.show()

## 5. Teammate Gaps

Teammate comparisons control partially for car performance. Smaller intra-team gaps indicate a tighter pairing, while large gaps suggest driver-level or race-execution separation.

In [15]:
driver_team = drivers[["session_key", "driver_number", "full_name", "team_name"]].drop_duplicates(["session_key", "driver_number"])
team_results = session_result.merge(driver_team, on=["session_key", "driver_number"], how="left")
team_results["finish_pos"] = pd.to_numeric(team_results["position"], errors="coerce")
team_results = team_results.dropna(subset=["finish_pos", "team_name"])
team_gap_rows = []
for (session_key, team_name), group in team_results.groupby(["session_key", "team_name"]):
    if len(group) == 2:
        ordered = group.sort_values("finish_pos")
        team_gap_rows.append({
            "session_key": session_key,
            "team_name": team_name,
            "best_driver_number": int(ordered.iloc[0]["driver_number"]),
            "best_driver": ordered.iloc[0]["full_name"],
            "second_driver_number": int(ordered.iloc[1]["driver_number"]),
            "second_driver": ordered.iloc[1]["full_name"],
            "gap_positions": float(ordered.iloc[1]["finish_pos"] - ordered.iloc[0]["finish_pos"]),
        })
team_gaps = pd.DataFrame(team_gap_rows)
team_gap_summary = team_gaps.groupby("team_name", as_index=False).agg(
    avg_gap=("gap_positions", "mean"),
    median_gap=("gap_positions", "median"),
    paired_sessions=("session_key", "count"),
).sort_values("avg_gap")
team_gaps.to_csv(OUTPUT_TABLES / "teammate_gap_events.csv", index=False)
team_gap_summary.to_csv(OUTPUT_TABLES / "teammate_gap_summary.csv", index=False)
display(team_gap_summary)

,team_name,avg_gap,median_gap,paired_sessions
2,Audi,2.000000,2.0,3
3,Cadillac,2.400000,2.0,5
4,Ferrari,2.967742,2.0,62
6,Kick Sauber,3.212766,3.0,47
7,McLaren,3.316667,2.0,60
8,Mercedes,3.406780,3.0,59
0,Alpine,3.509434,2.0,53
5,Haas F1 Team,3.931034,3.0,58
12,Williams,4.042553,3.0,47
9,RB,4.080000,5.0,25


In [16]:
fig = px.bar(
    team_gap_summary,
    x="team_name",
    y="avg_gap",
    color="paired_sessions",
    title="Average Intra-Team Finish Gap",
    labels={"avg_gap": "Average finish-position gap", "team_name": "Team"},
)
fig.update_xaxes(tickangle=35)
fig.write_html(OUTPUT_CHARTS / "teammate_gap_summary.html", include_plotlyjs="cdn")
fig.show()

## 6. Season Trends

This section tracks whether driver outcomes improve or decline across the available 2024-2026 sample.

In [17]:
driver_year = (
    results.dropna(subset=["finish_pos"])
    .groupby(["driver_id", "year"], as_index=False)
    .agg(
        avg_finish=("finish_pos", "mean"),
        points=("points", "sum"),
        starts=("session_key", "nunique"),
        wins=("finish_pos", lambda s: int((s == 1).sum())),
        primary_driver_number=("driver_number", lambda s: int(s.mode().iloc[0]) if not s.mode().empty else int(s.iloc[0])),
        primary_team=("team_name", lambda s: s.dropna().mode().iloc[0] if not s.dropna().mode().empty else "Unknown"),
    )
)
driver_year = driver_year[driver_year["starts"] >= 3].sort_values(["driver_id", "year"])
driver_year["prev_avg_finish"] = driver_year.groupby("driver_id")["avg_finish"].shift(1)
driver_year["finish_improvement"] = driver_year["prev_avg_finish"] - driver_year["avg_finish"]
driver_year.to_csv(OUTPUT_TABLES / "driver_year_trends.csv", index=False)
display(driver_year.head(30))

,driver_id,year,avg_finish,points,starts,wins,primary_driver_number,primary_team,prev_avg_finish,finish_improvement
0,Alexander ALBON,2024,13.260870,12.0,23,0,23,Williams,NaN,NaN
1,Alexander ALBON,2025,10.153846,73.0,26,0,23,Williams,13.260870,3.107023
2,Alexander ALBON,2026,15.833333,1.0,6,0,23,Williams,10.153846,-5.679487
3,Arvid LINDBLAD,2026,11.200000,5.0,5,0,41,Racing Bulls,NaN,NaN
4,Carlos SAINZ,2024,4.629630,290.0,27,2,55,Ferrari,NaN,NaN
5,Carlos SAINZ,2025,10.560000,64.0,25,0,55,Williams,4.629630,-5.930370
6,Carlos SAINZ,2026,11.500000,6.0,8,0,55,Williams,10.560000,-0.940000
7,Charles LECLERC,2024,3.965517,356.0,29,3,16,Ferrari,NaN,NaN
8,Charles LECLERC,2025,5.307692,242.0,26,0,16,Ferrari,3.965517,-1.342175
9,Charles LECLERC,2026,4.000000,75.0,8,0,16,Ferrari,5.307692,1.307692


In [18]:
season_avg = driver_year.groupby("year", as_index=False).agg(avg_finish=("avg_finish", "mean"), avg_points=("points", "mean"))
fig = px.line(season_avg, x="year", y="avg_finish", markers=True, title="Average Driver Finish by Season")
fig.update_yaxes(autorange="reversed")
fig.write_html(OUTPUT_CHARTS / "season_average_finish.html", include_plotlyjs="cdn")
fig.show()

In [19]:
improvement = driver_year.dropna(subset=["finish_improvement"]).copy()
fig = px.histogram(
    improvement,
    x="finish_improvement",
    nbins=25,
    title="Year-over-Year Finish Position Improvement",
    labels={"finish_improvement": "Improvement in average finish position"},
)
fig.add_vline(x=0, line_dash="dash")
fig.write_html(OUTPUT_CHARTS / "driver_improvement_distribution.html", include_plotlyjs="cdn")
fig.show()

## Final Driver Analysis Report

This report captures the headline driver-level signals available for Gold feature engineering.

In [20]:
report = {
    "notebook": NOTEBOOK_NAME,
    "timestamp": datetime.now().isoformat(),
    "total_drivers": int(results["driver_id"].nunique()),
    "sessions_analyzed": int(sessions["session_key"].nunique()),
    "grand_prix_sessions": int((sessions["event_type"] == "GRAND_PRIX_RACE").sum()),
    "sprint_sessions": int((sessions["event_type"] == "SPRINT_RACE").sum()),
    "mean_win_rate_pct_min_5_starts": float(win_rates.mean()) if len(win_rates) else 0.0,
    "top3_win_concentration_pct": top3_pct,
    "lap_consistency_finish_corr": consistency_corr,
    "total_overtakes": int(len(overtakes)),
    "grid_finish_corr": grid_corr,
    "avg_teammate_gap": float(team_gaps["gap_positions"].mean()) if not team_gaps.empty else 0.0,
    "driver_year_rows": int(len(driver_year)),
}
write_report("driver_analysis", report)
write_insight(
    "Silver Driver Analysis Insights",
    [
        f"Analyzed {report['total_drivers']} drivers across {report['sessions_analyzed']} race/sprint sessions.",
        f"Top three drivers account for {report['top3_win_concentration_pct']:.1f}% of wins.",
        f"Grid-to-finish correlation is {report['grid_finish_corr']:.3f}.",
        f"Lap consistency to finish correlation is {report['lap_consistency_finish_corr']:.3f}.",
    ],
    [],
    [
        "Use driver_summary, consistency, overtake balance, and grid dynamics as Gold feature candidates.",
        "Use event_type to separate Grand Prix race and Sprint race behavior.",
        "Treat teammate gaps as contextual features because team pairing controls partially for car performance.",
    ],
)
(CHECKPOINTS / "silver_driver_analysis_completed.txt").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(report)

{'notebook': '02_driver_analysis', 'timestamp': '2026-06-02T00:46:00.689088', 'total_drivers': 28, 'sessions_analyzed': 70, 'grand_prix_sessions': 55, 'sprint_sessions': 15, 'mean_win_rate_pct_min_5_starts': 3.7388521787348092, 'top3_win_concentration_pct': 72.05882352941177, 'lap_consistency_finish_corr': -0.4978351339022036, 'total_overtakes': 13985, 'grid_finish_corr': 0.7415668342234891, 'avg_teammate_gap': 3.9711191335740073, 'driver_year_rows': 66}
